In [ ]:
import subprocess

def run(cmd):
    # Execute command and capture both stdout and stderr
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

    # Print the last 1000 characters of stdout
    print(result.stdout[-1000:] if result.stdout else "")

    # Print the last 500 characters of stderr if any errors occurred
    if result.stderr:
        print("STDERR:", result.stderr[-500:])

    return result.returncode

# Check GPU availability and memory
print("--- GPU Information ---")
run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")

# Check Python version
print("\n--- Python Version ---")
run("python --version")

# Check CUDA compiler version
print("\n--- CUDA (nvcc) Version ---")
run("nvcc --version")

# Check pip version
print("\n--- pip Version ---")
run("pip --version")

Tesla T4, 15360 MiB

Python 3.12.13

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0

pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)



0

In [ ]:
import subprocess

def run(cmd):
    # Print the first 70 characters of the command for logging
    print(f">>> {cmd[:70]}")
    result = subprocess.run(cmd, shell=True)

    # Status indicators for success (✅) or failure (❌)
    print(f"{'✅' if result.returncode == 0 else '❌'} rc={result.returncode}")
    return result.returncode

# Install uv (fast package manager) and MinerU
print("📦 Initializing package installation...")
run("pip install -q uv")
run("uv pip install -U 'mineru[all]'")

print("\n✅ Done. Please restart the session manually to apply changes.")

>>> pip install -q uv
✅ rc=0
>>> uv pip install -U 'mineru[all]'
✅ rc=0

✅ Done. Restart Session manually.


In [ ]:
import subprocess

# Verify if installation was successful
result = subprocess.run("mineru --version", shell=True, capture_output=True, text=True)
print(f"MinerU version: {result.stdout.strip()}")

# Download models (Required for first-time setup, takes ~5-10 minutes)
print("\n⏳ Downloading models...")
subprocess.run("mineru-models-download", shell=True)
print("✅ Models ready")

MinerU version: mineru, version 3.0.9

⏳ Downloading models...
✅ Models ready


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import subprocess
import shutil
from pathlib import Path
from tqdm import tqdm

PDF_DIR    = Path("/content/drive/MyDrive/Uchicago/Capstone/1/Assignment3/pdfs")
OUTPUT_DIR = Path("/content/drive/MyDrive/Uchicago/Capstone/1/Assignment3/parsed_output_mineru")
OUTPUT_DIR.mkdir(exist_ok=True)

# Target specific IDs for processing with MinerU
failed_ids = [
    '2410.10516v3', '2412.00651v1', '2411.18627v2', '2412.07587v6',
    '2411.04376v2', '2410.18929v2', '2410.20812v3', '2412.07042v1',
    '2410.13267v2', '2410.07168v2', '2411.11853v3', '2501.01454v2',
    '2412.12783v2', '2411.00816v2', '2411.08777v2', '2410.08642v2',
    '2411.04950v4', '2411.05000v2'
]

TEMP_DIR = Path("/tmp/mineru_temp")
results = {}
failed  = []

for arxiv_id in tqdm(failed_ids, desc="Processing with MinerU"):
    pdf_path = PDF_DIR / f"{arxiv_id}.pdf"

    if not pdf_path.exists():
        print(f"⚠️  PDF not found: {arxiv_id}")
        failed.append(arxiv_id)
        continue

    doc_dir = OUTPUT_DIR / arxiv_id
    out_path = doc_dir / f"{arxiv_id}.md"

    # Skip if already processed
    if out_path.exists():
        print(f"⏭️  Skip: {arxiv_id}")
        results[arxiv_id] = {"status": "skipped"}
        continue

    doc_dir.mkdir(exist_ok=True)
    TEMP_DIR.mkdir(exist_ok=True)

    try:
        # Execute MinerU CLI pipeline
        cmd = f"mineru -p '{pdf_path}' -o '{TEMP_DIR}' -b pipeline"
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

        if result.returncode != 0:
            raise RuntimeError(result.stderr[-500:])

        # MinerU outputs to TEMP_DIR/{arxiv_id}/
        temp_out = TEMP_DIR / arxiv_id
        md_files = list(temp_out.rglob("*.md"))
        img_files = list(temp_out.rglob("images"))

        if not md_files:
            raise RuntimeError("No markdown output found")

        # Copy markdown file to final destination
        md_src = md_files[0]
        shutil.copy(md_src, out_path)

        # Copy extracted images folder
        img_dir = doc_dir / "images"
        for img_folder in img_files:
            if img_folder.is_dir():
                if img_dir.exists():
                    shutil.rmtree(img_dir)
                shutil.copytree(img_folder, img_dir)

        text = out_path.read_text(encoding="utf-8")
        img_count = len(list(img_dir.glob("*"))) if img_dir.exists() else 0

        results[arxiv_id] = {
            "chars":  len(text),
            "images": img_count,
            "status": "success"
        }
        print(f"✅ {arxiv_id}: {len(text)} chars, {img_count} images")

    except Exception as e:
        print(f"❌ {arxiv_id}: {e}")
        failed.append(arxiv_id)
    finally:
        # Cleanup temporary files
        if TEMP_DIR.exists():
            shutil.rmtree(TEMP_DIR)

import json
# Generate final execution report
report = {
    "total":   len(failed_ids),
    "success": len([v for v in results.values() if v.get("status") == "success"]),
    "skipped": len([v for v in results.values() if v.get("status") == "skipped"]),
    "failed":  failed,
}
with open(OUTPUT_DIR / "_report_mineru.json", "w") as f:
    json.dump(report, f, indent=2)

print(f"\n✅ Success : {report['success']}")
print(f"⏭️  Skipped : {report['skipped']}")
print(f"❌ Failed  : {len(failed)}")
if failed:
    print(f"Failed IDs : {failed}")

Processing with MinerU:   6%|▌         | 1/18 [02:20<39:43, 140.20s/it]

✅ 2410.10516v3: 110536 chars, 46 images


Processing with MinerU:  11%|█         | 2/18 [03:58<30:45, 115.36s/it]

✅ 2412.00651v1: 110283 chars, 32 images


Processing with MinerU:  17%|█▋        | 3/18 [05:08<23:40, 94.67s/it] 

...

✅ 2410.08642v2: 67630 chars, 11 images


Processing with MinerU:  94%|█████████▍| 17/18 [24:46<01:27, 87.07s/it]

✅ 2411.04950v4: 112935 chars, 40 images


Processing with MinerU: 100%|██████████| 18/18 [26:17<00:00, 87.63s/it]

✅ 2411.05000v2: 84551 chars, 33 images

✅ Success : 18
⏭️  Skipped : 0
❌ Failed  : 0
